##### Import statements:

In [1]:
import socket
import os
import pathlib
import sys
import numpy as np
import pandas as pd
import multiprocessing as mp
from multiprocessing import set_start_method
from datetime import datetime
import time
from sklearn.linear_model import LogisticRegression
from ws.test import init_par, fcn_inner, fcn_outer
from analysis_metadata.analysis_metadata import Metadata, write_metadata, increment_dir_name

hostname = socket.gethostname()
set_start_method('spawn')

##### Define params:

In [13]:
n_iter = 5000
n_feat = 5
n_obs = int(10**4)
n_cores = 5

# Output params:
save_output = True
if 'rc.zi.columbia.edu' in hostname:
    base_output_directory = os.path.join('/', 'mnt', 'smb', 'locker', 'issa-locker', 'users', 'Dan', 'code', 'ws', 'results', 'speed_tests')

##### Main:

In [14]:
if __name__ == '__main__':
    start = time.time()
    pool = mp.Pool(processes=n_cores, initializer=init_par, initargs=(0,))
    pool_output = [pool.apply_async(fcn_outer,args=(n_feat, n_obs)) for x in n_obs*np.ones(n_iter).astype(int)]
    pool_output = [p.get() for p in pool_output]
    durs = [p[1] for p in pool_output]
    pool.close()
    stop = time.time()

##### Print results:

In [15]:
if __name__ == '__main__':
    total_dur = stop - start
    mean_fcn_call_dur = np.mean(durs)
    
    print('total dur = {} s'.format(total_dur))
    print('mean dur per call to fcn() = {} s'.format(mean_fcn_call_dur))

total dur = 17.068914651870728 s
mean dur per call to fcn() = 0.011108099985122681 s


##### Save output if requested:

In [16]:
if __name__ == '__main__' and save_output:

    # Create output directory if necessary:
    if not os.path.exists(base_output_directory):
        pathlib.Path(base_output_directory).mkdir(parents=True, exist_ok=True)
    curr_output_directory=increment_dir_name(base_output_directory, 'run')
    if not os.path.exists(curr_output_directory):
        pathlib.Path(curr_output_directory).mkdir(parents=True, exist_ok=True)

    # Save results to metadata file:
    M = Metadata()
    script_path = os.path.join(os.getcwd(), 'mp_speed_test.ipynb')
    results_path = os.path.join(curr_output_directory, 'speed_test_results.json')
    M.add_input(script_path)
    M.add_param('hostname', hostname)
    M.add_param('n_iter', n_iter)
    M.add_param('n_feat', n_feat)
    M.add_param('n_obs', n_obs)
    M.add_param('n_cores', n_cores)
    M.add_param('mean_fcn_call_dur', mean_fcn_call_dur)
    M.add_output(results_path)
    now = datetime.now()
    M.date = now.strftime('%Y-%m-%d')
    M.time = now.strftime('%H:%M:%S')
    M.duration = total_dur
    write_metadata(M, results_path, debug=True)